In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist
import os
import warnings
import time
from datetime import datetime
import openpyxl

# غیرفعال کردن هشدارهای غیرضروری
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: تعریف تمام توابع تحلیل (4 تابع مجزا)
# ============================================================================

def analysis_1_turbine_comprehensive(file_path, output_filename):
    """
    تحلیل جامع توربین - کد شماره 1
    شامل: DBSCAN + 3-Sigma + Partial Correlation + RCA
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 1 (توربین - جامع)")
    print(f"{'='*60}")
    
    # لیست تمام سنسورها برای تحلیل شبکه ارتباطی
    all_features = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361', 
                    'AssetID_9368', 'AssetID_9369', 'AssetID_9370',
                    'AssetID_9343', 'AssetID_9344', 'AssetID_9408']
    
    # سنسورهایی که تست ۳-سیگما روی آن‌ها اجرا می‌شود
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    try:
        df_raw = pd.read_excel(file_path)
        df_raw['date'] = pd.to_datetime(df_raw['date'])
        df_raw = df_raw.sort_values(by='date')
        print(f"✅ دیتا با موفقیت بارگذاری شد. تعداد رکوردها: {len(df_raw):,}")
        print(f"📅 بازه زمانی: {df_raw['date'].min()} تا {df_raw['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    # پیش‌پردازش و حذف داده‌های پرت (DBSCAN - حذف ۱۰ درصد داده‌های دور)
    print("🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...")
    
    scaler = StandardScaler()
    scaled_data = scaler.fit_transform(df_raw[all_features])
    
    dbscan = DBSCAN(eps=0.5, min_samples=5)
    labels = dbscan.fit_predict(scaled_data)
    
    cluster_centers = {cid: scaled_data[labels == cid].mean(axis=0) for cid in set(labels) if cid != -1}
    
    def calculate_distance(i):
        label, point = labels[i], scaled_data[i].reshape(1, -1)
        if label != -1:
            return cdist(point, cluster_centers[label].reshape(1, -1))[0][0]
        return np.min(cdist(point, np.array(list(cluster_centers.values())))) if cluster_centers else 0.0
    
    df_raw['distance'] = [calculate_distance(i) for i in range(len(df_raw))]
    
    before_count = len(df_raw)
    df_cleaned = df_raw.sort_values(by='distance', ascending=False).iloc[int(len(df_raw)*0.1):].copy()
    df_cleaned = df_cleaned.sort_values(by='date').set_index('date')
    after_count = len(df_cleaned)
    print(f"   حذف {before_count - after_count:,} ردیف به عنوان داده‌های پرت")
    
    # جداسازی بازه نرمال (Baseline) و بازه تحلیل (Fault)
    print("🔄 مرحله 2: جداسازی بازه‌های نرمال و تحلیل...")
    
    last_date = df_cleaned.index.max()
    start_analysis_date = last_date - pd.Timedelta(days=30)
    baseline_end = start_analysis_date - pd.Timedelta(days=1)
    baseline_start = baseline_end - pd.Timedelta(days=30)
    
    df_baseline = df_cleaned[(df_cleaned.index >= baseline_start) & (df_cleaned.index <= baseline_end)]
    df_fault = df_cleaned[df_cleaned.index >= start_analysis_date].copy()
    
    print(f"   بازه نرمال: {baseline_start} تا {baseline_end}")
    print(f"   بازه تحلیل: {start_analysis_date} تا {last_date}")
    print(f"   تعداد رکوردهای بازه نرمال: {len(df_baseline):,}")
    print(f"   تعداد رکوردهای بازه تحلیل: {len(df_fault):,}")
    
    # تست چندسطحی انحراف معیار (3-Sigma Rule)
    print("🔄 مرحله 3: اجرای تست 3-Sigma...")
    
    def get_sigma_status(val, mean, std):
        if std == 0: return 'Normal'
        deviation = abs(val - mean) / std
        if deviation > 3:
            return 'Action Required'
        elif deviation > 2:
            return 'Warning'
        elif deviation > 1:
            return 'Normal (Minor Change)'
        else:
            return 'Normal'
    
    for col in target_sensors:
        m = df_baseline[col].mean()
        s = df_baseline[col].std() if df_baseline[col].std() != 0 else 1e-6
        df_fault[f'Status_{col}'] = df_fault[col].apply(lambda x: get_sigma_status(x, m, s))
    
    for col in target_sensors:
        status_counts = df_fault[f'Status_{col}'].value_counts()
        print(f"   {col}: {dict(status_counts)}")
    
    # محاسبات همبستگی جزئی (Partial Correlation)
    print("🔄 مرحله 4: محاسبات همبستگی جزئی...")
    
    def get_partial_corr(data, columns):
        corr_matrix = data[columns].corr().values
        precision = np.linalg.inv(corr_matrix + np.eye(corr_matrix.shape[0])*1e-6)
        d = np.sqrt(np.diag(precision))
        partial_corr = -precision / np.outer(d, d)
        np.fill_diagonal(partial_corr, 1.0)
        return pd.DataFrame(partial_corr, index=columns, columns=columns)
    
    pcorr_baseline = get_partial_corr(df_baseline, all_features)
    pcorr_fault = get_partial_corr(df_fault[all_features], all_features)
    delta_pcorr = pcorr_fault - pcorr_baseline
    
    print("   ✅ محاسبات همبستگی انجام شد")
    
    # محاسبه امتیازات RCA و رتبه‌بندی ریشه خطا
    print("🔄 مرحله 5: محاسبه امتیازات RCA...")
    
    deviation_scores = {c: abs((df_fault[c].mean() - df_baseline[c].mean()) / (df_baseline[c].std() or 1e-6)) for c in all_features}
    change_scores = {c: np.mean([abs(delta_pcorr.loc[c, o]) for o in all_features if o != c]) for c in all_features}
    
    def normalize_dict(d):
        vals = np.array(list(d.values()))
        if vals.max() == vals.min(): return {k: 0.5 for k in d.keys()}
        return {k: (v - vals.min()) / (vals.max() - vals.min()) for k, v in d.items()}
    
    dev_norm = normalize_dict(deviation_scores)
    chg_norm = normalize_dict(change_scores)
    
    rca_list = []
    for c in all_features:
        strength_fault = np.mean([abs(pcorr_fault.loc[c, o]) for o in all_features if o != c])
        score = (dev_norm[c]*0.5) + (chg_norm[c]*0.5)
        
        rca_list.append({
            'Sensor': c,
            'Change_Normalized': chg_norm[c],
            'Change_Score_Delta': change_scores[c],
            'Deviation_Normalized': dev_norm[c],
            'Deviation_Score': deviation_scores[c],
            'Strength_Score_Fault': strength_fault,
            'RCA_Score': score,
            'Root_Cause_Suspicion': score*(1 - strength_fault)
        })
    
    df_rca_summary = pd.DataFrame(rca_list).sort_values(by='Root_Cause_Suspicion', ascending=False)
    
    # ذخیره‌سازی در اکسل با شیت‌های مجزا
    print("💾 مرحله 6: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            status_cols = [f'Status_{c}' for c in target_sensors]
            df_fault[target_sensors + status_cols].reset_index().to_excel(writer, sheet_name='Fault_Status', index=False)
            df_rca_summary.to_excel(writer, sheet_name='RCA_Summary', index=False)
            delta_pcorr.to_excel(writer, sheet_name='Delta_Correlation_Matrix')
        
        print(f"✅ فایل با موفقیت ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_fault):,}")
        
        print("\n--- ۵ مظنون اصلی خرابی ---")
        print(df_rca_summary[['Sensor', 'RCA_Score', 'Root_Cause_Suspicion']].head(5))
        
        return True
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return False


def analysis_2_turbine_ewma(file_path, output_filename):
    """
    تحلیل EWMA توربین - کد شماره 2
    شامل: EWMA + کنترل حدود + RCA
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 2 (توربین - EWMA)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    LAMBDA = 0.2
    
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل یافت نشد در مسیر: {file_path}")
        return False
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df.set_index('date', inplace=True)
        print("✅ مرحله ۱: فایل بارگذاری شد.")
        print(f"📅 بازه زمانی داده‌ها: {df.index.min()} تا {df.index.max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return False
    
    # جداسازی بازه سلامت و بازه یک ماه اخیر
    try:
        last_date = df.index.max()
        split_date = last_date - pd.Timedelta(days=30)
        baseline_start = split_date - pd.Timedelta(days=30)
        
        df_baseline = df.loc[baseline_start:split_date].copy()
        df_fault = df.loc[split_date:last_date].copy()
        
        print(f"📊 بازه سلامت: {baseline_start.date()} تا {split_date.date()}")
        print(f"⚠️ بازه خطا: {split_date.date()} تا {last_date.date()}")
    except Exception as e:
        print(f"❌ خطا در پردازش تاریخ‌ها: {e}")
        return False
    
    print("⏳ در حال محاسبه شاخص‌ها...")
    print(f"🔧 لاندا (λ) = {LAMBDA}")
    print("="*80)
    
    target_analysis_results = []
    
    def calculate_ewma(data, lambda_val):
        ewma_values = np.zeros(len(data))
        if len(data) == 0:
            return ewma_values
        ewma_values[0] = data[0]
        for t in range(1, len(data)):
            ewma_values[t] = lambda_val * data[t] + (1 - lambda_val) * ewma_values[t-1]
        return ewma_values
    
    def calculate_control_limits(mean, std, lambda_val):
        factor = 3 * std * np.sqrt(lambda_val / (2 - lambda_val))
        ucl = mean + factor
        lcl = mean - factor
        return ucl, lcl
    
    for col in target_sensors:
        if col not in df_fault.columns:
            print(f"⚠️ سنسور {col} در داده‌ها وجود ندارد.")
            continue
        
        if col not in df_baseline.columns:
            print(f"⚠️ سنسور {col} در داده‌های بیس‌لاین وجود ندارد.")
            continue
        
        try:
            mean_base = df_baseline[col].mean()
            std_base = df_baseline[col].std()
            
            ucl, lcl = calculate_control_limits(mean_base, std_base, LAMBDA)
            
            fault_data = df_fault[col].values
            ewma_values = calculate_ewma(fault_data, LAMBDA)
            current_ewma = ewma_values[-1]
            
            time_diff_series = df_fault.index.to_series().diff().dt.total_seconds() / 3600.0
            val_diff = df_fault[col].diff()
            instant_slopes = val_diff / time_diff_series
            avg_slope = instant_slopes.mean()
            
            current_raw_value = df_fault[col].iloc[-1]
            
            if avg_slope <= 0:
                hours_to_ucl = 0.0
                reason = "شیب منفی یا صفر"
            elif current_raw_value >= ucl:
                hours_to_ucl = 0.0
                reason = "مقدار فعلی از UCL بیشتر یا مساوی است"
            else:
                hours_to_ucl = (ucl - current_raw_value) / avg_slope
                reason = f"محاسبه شد: ({ucl:.4f} - {current_raw_value:.4f}) / {avg_slope:.6f}"
            
            if np.isnan(hours_to_ucl) or np.isinf(hours_to_ucl):
                hours_to_ucl = 0.0
                reason = "مقدار نامعتبر (NaN/Inf)"
            else:
                hours_to_ucl = round(hours_to_ucl, 2)
            
            print(f"\n🔍 {col}:")
            print(f"   مقدار فعلی: {current_raw_value:.4f}")
            print(f"   UCL: {ucl:.4f}")
            print(f"   شیب متوسط: {avg_slope:.8f}")
            print(f"   Hours_to_UCL: {hours_to_ucl} ساعت ← {reason}")
            
            out_of_control = "No"
            if current_ewma > ucl or current_ewma < lcl:
                out_of_control = "Yes ⚠️"
            
            target_analysis_results.append({
                'Sensor': col,
                'Current_Raw_Value': round(current_raw_value, 4),
                'Current_EWMA': round(current_ewma, 4),
                'UCL': round(ucl, 4),
                'LCL': round(lcl, 4),
                'Baseline_Mean': round(mean_base, 4),
                'Baseline_Std': round(std_base, 4),
                'Average_Slope_per_Hour': round(avg_slope, 6),
                'Hours_to_UCL': hours_to_ucl,
                'Out_Of_Control': out_of_control
            })
            
        except Exception as e:
            print(f"❌ خطا در پردازش سنسور {col}: {e}")
            continue
    
    # رتبه‌بندی RCA
    print("\n" + "="*80)
    print("⏳ در حال محاسبه رتبه‌بندی RCA...")
    
    numeric_cols = df_baseline.select_dtypes(include=[np.number]).columns.tolist()
    rca_list = []
    for c in numeric_cols:
        if c not in df_fault.columns:
            continue
        try:
            mean_base = df_baseline[c].mean()
            std_base = df_baseline[c].std()
            mean_fault = df_fault[c].mean()
            
            if std_base > 0:
                deviation_score = abs(mean_fault - mean_base) / std_base
            else:
                deviation_score = 0
                
            rca_list.append({
                'Sensor': c, 
                'Baseline_Mean': round(mean_base, 4),
                'Fault_Mean': round(mean_fault, 4),
                'Deviation_Score': round(deviation_score, 4)
            })
        except Exception as e:
            continue
    
    rca_summary = pd.DataFrame(rca_list).sort_values('Deviation_Score', ascending=False)
    
    # ذخیره در اکسل
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            pd.DataFrame(target_analysis_results).to_excel(writer, sheet_name='Speed_Analysis', index=False)
            rca_summary.to_excel(writer, sheet_name='RCA_Ranking', index=False)
            
            methodology_note = pd.DataFrame({
                'Parameter': [
                    'Lambda (λ)', 
                    'UCL Formula', 
                    'LCL Formula', 
                    'EWMA Formula', 
                    'Hours_to_UCL Rule'
                ],
                'Value': [
                    f'{LAMBDA}', 
                    f'μ + 3σ√(λ/(2-λ))', 
                    f'μ - 3σ√(λ/(2-λ))', 
                    'E_t = λ·X_t + (1-λ)·E_{t-1}',
                    'If slope>0 and value<UCL → (UCL-value)/slope | Else → 0'
                ]
            })
            methodology_note.to_excel(writer, sheet_name='Methodology', index=False)
        
        print(f"\n🚀 گزارش با موفقیت ساخته شد:\n{output_filename}")
        
        # آمار نهایی
        positive_slopes = [r for r in target_analysis_results if r['Average_Slope_per_Hour'] > 0]
        positive_hours = [r for r in target_analysis_results if r['Hours_to_UCL'] > 0]
        
        print(f"\n📊 آمار نهایی Hours_to_UCL:")
        print(f"   └─ کل سنسورهای هدف: {len(target_analysis_results)}")
        print(f"   └─ سنسورهای با شیب مثبت: {len(positive_slopes)}")
        print(f"   └─ سنسورهای با زمان مثبت (در حال رسیدن): {len(positive_hours)}")
        print(f"   └─ سنسورهای با زمان صفر: {len(target_analysis_results) - len(positive_hours)}")
        
        if positive_hours:
            print(f"\n📈 سنسورهایی که در حال رسیدن به UCL هستند:")
            for res in positive_hours:
                print(f"   └─ {res['Sensor']}: {res['Hours_to_UCL']} ساعت دیگر (شیب={res['Average_Slope_per_Hour']:.6f})")
        
        return True
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return False


def analysis_3_turbine_ewma_dbscan(file_path, output_filename):
    """
    تحلیل EWMA با پاکسازی DBSCAN - کد شماره 3
    شامل: EWMA + DBSCAN برای پاکسازی بیس‌لاین
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 3 (توربین - EWMA با DBSCAN)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    if not os.path.exists(file_path):
        print(f"❌ فایل یافت نشد: {file_path}")
        return False
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ دیتا با موفقیت بارگذاری شد. تعداد رکوردها: {len(df):,}")
        print(f"📅 بازه زمانی: {df['date'].min()} تا {df['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن فایل: {e}")
        return False
    
    # تعریف بازه‌های زمانی
    last_date = df['date'].max()
    fault_start = last_date - pd.Timedelta(days=30)
    baseline_start = fault_start - pd.Timedelta(days=30)
    
    print(f"📅 بازه نرمال: {baseline_start} تا {fault_start}")
    print(f"📅 بازه تحلیل: {fault_start} تا {last_date}")
    
    df_baseline = df[(df['date'] >= baseline_start) & (df['date'] < fault_start)].copy()
    df_fault = df[df['date'] >= fault_start].copy()
    
    print(f"📊 تعداد رکوردهای بازه نرمال: {len(df_baseline):,}")
    print(f"📊 تعداد رکوردهای بازه تحلیل: {len(df_fault):,}")
    
    # بخش اول: آماده‌سازی تب EWMA
    print("🔄 مرحله 1: محاسبات EWMA...")
    
    def remove_outliers_dbscan(series):
        if series.empty: return series
        X = series.values.reshape(-1, 1)
        dbscan = DBSCAN(eps=series.std()*0.5, min_samples=5)
        clusters = dbscan.fit_predict(X)
        return series[clusters != -1]
    
    ewma_list = []
    baseline_stats = []
    
    for col in target_sensors:
        if col not in df.columns:
            print(f"⚠️ هشدار: ستون {col} در داده‌ها وجود ندارد")
            continue
        
        temp_fault = df_fault[['date', col]].copy()
        temp_fault = temp_fault.rename(columns={col: 'Value'})
        temp_fault['AssetID'] = col
        temp_fault['EWMA'] = temp_fault['Value'].ewm(alpha=0.2, adjust=False).mean()
        ewma_list.append(temp_fault[['date', 'AssetID', 'Value', 'EWMA']])
        
        if col in df_baseline.columns:
            clean_baseline = remove_outliers_dbscan(df_baseline[col].dropna())
            
            if not clean_baseline.empty:
                mean_val = clean_baseline.mean()
                std_val = clean_baseline.std()
                
                baseline_stats.append({
                    'AssetID': col,
                    'Clean_Mean': round(mean_val, 4),
                    'Clean_Std': round(std_val, 4),
                    'Normal_Upper_Limit(3Sigma)': round(mean_val + 3*std_val, 4),
                    'Outliers_Removed': len(df_baseline[col]) - len(clean_baseline)
                })
                
                print(f"   {col}: میانگین={mean_val:.4f}, انحراف معیار={std_val:.4f}, حذف {len(df_baseline[col]) - len(clean_baseline)} داده پرت")
    
    # ذخیره در اکسل
    print("💾 مرحله 2: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            if ewma_list:
                df_ewma_final = pd.concat(ewma_list, ignore_index=True)
                df_ewma_final.to_excel(writer, sheet_name='EWMA_Comparison', index=False)
                print(f"   ✅ تب EWMA_Comparison با {len(df_ewma_final):,} رکورد ذخیره شد")
            
            if baseline_stats:
                df_stats_final = pd.DataFrame(baseline_stats)
                df_stats_final.to_excel(writer, sheet_name='Baseline_Stats', index=False)
                print(f"   ✅ تب Baseline_Stats با {len(df_stats_final)} سنسور ذخیره شد")
        
        print(f"✅ فایل با موفقیت ذخیره شد: {output_filename}")
        return True
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return False


def analysis_4_turbine_dual_ewma(file_path, output_filename):
    """
    تحلیل Dual EWMA (روزانه/هفتگی) - کد شماره 4
    شامل: Dual EWMA با Gap و Deviation Percent
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 4 (توربین - Dual EWMA)")
    print(f"{'='*60}")
    
    target_sensors = ['AssetID_9358', 'AssetID_9359', 'AssetID_9360', 'AssetID_9361']
    
    if not os.path.exists(file_path):
        print(f"❌ خطا: فایل در مسیر زیر یافت نشد:\n{file_path}")
        return False
    
    try:
        df = pd.read_excel(file_path, parse_dates=['date'])
        df = df.sort_values('date')
        print(f"✅ مرحله ۱: داده‌ها بارگذاری شدند. تعداد رکوردها: {len(df):,}")
        print(f"📅 بازه زمانی: {df['date'].min()} تا {df['date'].max()}")
    except Exception as e:
        print(f"❌ خطا در خواندن اکسل: {e}")
        return False
    
    last_date = df['date'].max()
    one_month_ago = last_date - pd.Timedelta(days=30)
    df_recent = df[df['date'] >= one_month_ago].copy()
    
    print(f"📅 بازه تحلیل (یک ماه اخیر): {one_month_ago} تا {last_date}")
    print(f"📊 تعداد رکوردهای بازه تحلیل: {len(df_recent):,}")
    
    alpha_fast = 0.22
    alpha_slow = 0.035
    
    results_list = []
    print(f"⏳ مرحله ۲: تحلیل روند - سریع (Alpha={alpha_fast}) و کند (Alpha={alpha_slow})")
    
    for col in target_sensors:
        if col in df_recent.columns:
            temp_df = df_recent[['date', col]].copy()
            temp_df = temp_df.rename(columns={col: 'Raw_Value'})
            temp_df['AssetID'] = col
            temp_df['Daily_EWMA_Fast'] = temp_df['Raw_Value'].ewm(alpha=alpha_fast, adjust=False).mean()
            temp_df['Weekly_EWMA_Slow'] = temp_df['Raw_Value'].ewm(alpha=alpha_slow, adjust=False).mean()
            temp_df['Signal_Gap'] = temp_df['Daily_EWMA_Fast'] - temp_df['Weekly_EWMA_Slow']
            temp_df['Deviation_Percent'] = (temp_df['Signal_Gap'] / temp_df['Weekly_EWMA_Slow']) * 100
            
            cols_order = ['date', 'AssetID', 'Raw_Value', 'Daily_EWMA_Fast', 'Weekly_EWMA_Slow', 'Signal_Gap', 'Deviation_Percent']
            results_list.append(temp_df[cols_order])
            print(f"   ✅ {col}: {len(temp_df):,} رکورد پردازش شد")
    
    if not results_list:
        print("⚠️ هشدار: سنسورهای مورد نظر در فایل یافت نشدند.")
        return False
    
    print("🔄 مرحله ۳: تجمیع نتایج...")
    df_final_output = pd.concat(results_list, ignore_index=True)
    print(f"   ✅ تعداد کل رکوردها: {len(df_final_output):,}")
    
    print("💾 مرحله ۴: ذخیره خروجی...")
    
    try:
        os.makedirs(os.path.dirname(output_filename), exist_ok=True)
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df_final_output.to_excel(writer, index=False, sheet_name='Maintenance_Strategy')
        
        print(f"✅ فایل با موفقیت ذخیره شد: {output_filename}")
        print(f"📊 تعداد رکوردهای نهایی: {len(df_final_output):,}")
        return True
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل اکسل: {e}")
        return False


# ============================================================================
# بخش 2: تعریف وظایف (Jobs) با مسیرهای ورودی و خروجی
# ============================================================================

def get_analysis_jobs():
    """
    تعریف ۴ وظیفه تحلیل با مسیرهای ورودی و خروجی مربوطه
    """
    jobs = [
        {
            'name': 'Analysis 1 - Turbine Comprehensive',
            'function': analysis_1_turbine_comprehensive,
            'file_path': r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output1.xlsx'
        },
        {
            'name': 'Analysis 2 - Turbine EWMA',
            'function': analysis_2_turbine_ewma,
            'file_path': r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output2.xlsx'
        },
        {
            'name': 'Analysis 3 - Turbine EWMA with DBSCAN',
            'function': analysis_3_turbine_ewma_dbscan,
            'file_path': r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output3.xlsx'
        },
        {
            'name': 'Analysis 4 - Turbine Dual EWMA',
            'function': analysis_4_turbine_dual_ewma,
            'file_path': r'second_stage_inputs\G11\dsas_g11_turbine_bearings_output.xlsx',
            'output_filename': r'outputs\G11\dsas_g11_bearings_vibration_temp_univariate\univariate\dsas_g11_univariate_output4.xlsx'
        }
    ]
    return jobs


def run_all_analyses():
    """
    اجرای تمام ۴ تحلیل به ترتیب
    """
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌ها در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    jobs = get_analysis_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# اجرای وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    for r in results:
        status = "✅" if r['success'] else "❌"
        print(f"   {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 3: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)
    """
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل جامع بیرینگ")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}  # ذخیره زمان‌های اجرا شده
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص (۹:۰۰ و ۲۱:۰۰)
            if current_time in ["11:02", "11:05"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    # اجرای همه تحلیل‌ها
                    results = run_all_analyses()
                    
                    # ثبت زمان اجرا
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    # ۶۰ ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(60)
            
            # هر ۳۰ ثانیه یکبار بررسی کن
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)


# ============================================================================
# بخش 4: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع تحلیل بیرینگ (۴ تحلیل یکپارچه)")
        print("="*80)
        print("📋 لیست تحلیلها:")
        print("   1. تحلیل جامع توربین (DBSCAN + 3-Sigma + Partial Correlation)")
        print("   2. تحلیل EWMA توربین (کنترل حدود + RCA)")
        print("   3. تحلیل EWMA توربین با پاکسازی DBSCAN")
        print("   4. تحلیل Dual EWMA توربین (روزانه/هفتگی)")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        # اجرای زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع تحلیل بیرینگ (۴ تحلیل یکپارچه)
📋 لیست تحلیلها:
   1. تحلیل جامع توربین (DBSCAN + 3-Sigma + Partial Correlation)
   2. تحلیل EWMA توربین (کنترل حدود + RCA)
   3. تحلیل EWMA توربین با پاکسازی DBSCAN
   4. تحلیل Dual EWMA توربین (روزانه/هفتگی)
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - تحلیل جامع بیرینگ
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-09 11:02:01

🚀 شروع اجرای همه تحلیل‌ها در 2026-07-09 11:02:01

################################################################################
# اجرای وظیفه 1 از 4: Analysis 1 - Turbine Comprehensive
################################################################################

🔄 شروع تحلیل شماره 1 (توربین - جامع)
✅ دیتا با موفقیت بارگذاری شد. تعداد رکوردها: 11,752
📅 بازه زمانی: 2021-03-16 05:33:48 تا 2026-05-04 05:16:35
🔄 مرحله 1: پیش‌پردازش و حذف داده‌های پرت...
   حذف 1,175 ردیف به عنوان داده‌های پرت
🔄